# Model 4: Random Forest

**Project:** DataCo Smart Supply Chain — Late Delivery Risk Classification  
**Course:** CS280/CS485 Introduction to Artificial Intelligence — Spring 2026  
**Notebook:** `05_model_rf.ipynb` — implements Section 5.4 of `docs.md`  

This notebook trains and evaluates a `RandomForestClassifier` using `RandomizedSearchCV` for
hyperparameter tuning. It loads the prepared data produced by Phase 0, tunes over the grid
defined in docs.md §5.4, produces the n_estimators vs. F1 curve and feature-importance
bar chart (top 15), and saves all results to `results/rf_results.pkl` for Phase 6.

In [ ]:
!pip install -q "pandas>=3.0.2"

## 1. Imports

All dependencies are standard sklearn / scientific Python — no installation required beyond
`requirements.txt`.

In [ ]:
# Step 1: import standard libraries for arrays, data frames, and plotting
# Step 2: import sklearn modules for the model, search, CV, and metrics
# Step 3: suppress convergence and sklearn future warnings for clean output
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    RandomizedSearchCV, StratifiedKFold, cross_val_score
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report,
    precision_recall_curve
)

warnings.filterwarnings('ignore')
np.random.seed(42)

print('Imports OK')

## 2. Load Prepared Data

We load exclusively from `artifacts/prepared_data.pkl` — the single source of truth produced
by Phase 0. **No raw CSV is accessed here**, and no preprocessing objects (scaler, encoder)
are refit. This guarantees identical train/test splits across all model phases.

In [ ]:
# Step 1: define paths to the input artifact and the output results file
# Step 2: open and unpickle the prepared data produced by Phase 0
# Step 3: extract train/test arrays, feature names, and metadata
# Step 4: print shapes and class balance to confirm correct load
ARTIFACT_PATH = './artifacts/prepared_data.pkl'
RESULTS_PATH  = './results/rf_results.pkl'

with open(ARTIFACT_PATH, 'rb') as f:
    data = pickle.load(f)

X_train       = data['X_train']
X_test        = data['X_test']
y_train       = data['y_train']
y_test        = data['y_test']
feature_names = data['feature_names']

print(f'X_train shape : {X_train.shape}')
print(f'X_test  shape : {X_test.shape}')
print(f'y_train class balance : {np.bincount(y_train)}')
print(f'y_test  class balance : {np.bincount(y_test)}')
print(f'Number of features    : {len(feature_names)}')

**Interpretation:** The shapes confirm we are working with the preprocessed, scaled
training and test sets from Phase 0. Although Random Forest is scale-invariant (tree splits
depend only on feature rank, not magnitude), the unified pipeline applies `RobustScaler`
for consistency across all five models. The class balance is checked to confirm stratified
splitting preserved the original distribution.

## 3. Lecture 4 Connection — Random Forest as a Regularized Ensemble

> **Out-of-class extension flag:** Random Forest is **not** named in Lecture 4 by name.
> It is included here as the **first of the two allowed extensions** (per the professor's rule:
> ≥ 3 in-class models + at most 2 extensions). The justification below ties every key
> hyperparameter directly to Lecture 4's framework — so that any team member can defend
> this choice under Q&A.

### 3.1 Core Idea — Bagging

A Random Forest trains **B** decision trees, each on a bootstrap sample $Z_b$ (sampling the
training set with replacement). At each node, only $m = \sqrt{p}$ randomly selected features
are considered for splitting. Final prediction by majority vote:

$$\hat{y} = \arg\max_{c} \sum_{b=1}^{B} \mathbf{1}[T_b(\mathbf{x}) = c]$$

### 3.2 Regularization — Tying `max_depth` and `min_samples_leaf` to Lecture 4's cost(h)

Lecture 4 introduces the regularization framework:

$$\text{cost}(h) = \text{loss}(h) + \lambda \cdot \text{complexity}(h)$$

In a decision tree, **complexity(h)** is measured by tree depth and leaf granularity:

| sklearn hyperparameter | Role in Lecture 4 formula | Effect |
|:---|:---|:---|
| `max_depth` | Controls complexity(h) — deeper tree = higher complexity | Deep trees fit training data perfectly (overfit); shallow trees generalise better |
| `min_samples_leaf` | Controls complexity(h) — large leaf minimum = smoother boundary | Large value prevents creating leaves that represent only a few points (reduces overfitting) |

Setting `max_depth=10` or `min_samples_leaf=10` is equivalent to raising λ in the lecture
formula: it penalises models that grow too complex and overfit the training set.

### 3.3 Variance Reduction — Tying Bagging to Lecture 4's Overfitting Discussion

Lecture 4 discusses overfitting as a model fitting the *noise* in the training set, not the
true underlying pattern. A single decision tree grown to full depth has **high variance**: it
memorises the training partition perfectly but generalises poorly.

Bagging (bootstrap aggregating) reduces variance by averaging across **B** trees trained on
slightly different data samples. Each tree overfits its own bootstrap sample, but because the
trees are decorrelated (different data + random feature subsets), their errors **cancel out**
in the average — the ensemble generalises better than any individual tree. This is a direct,
practical solution to the overfitting problem discussed in Lecture 4.

## 4. Hyperparameter Grid

The grid follows Section 5.4 of `docs.md` exactly:

| Hyperparameter | Search Range | Lecture 4 mapping |
|:---|:---|:---|
| `n_estimators` | {100, 200, 500} | More trees → lower variance (reduces overfitting) |
| `max_depth` | {None, 10, 20, 30} | Controls complexity(h) in cost(h) = loss(h) + λ·complexity(h) |
| `max_features` | {'sqrt', 'log2'} | Features per split — decorrelates trees |
| `min_samples_leaf` | {1, 5, 10} | Controls complexity(h) — larger = simpler leaves |

**Why RandomizedSearchCV?** The full cross-product (3 × 4 × 2 × 3 = 72 combinations) × 5 folds
= 360 model fits on 144K rows would take tens of minutes. `RandomizedSearchCV(n_iter=50, cv=5)`
samples 50 random combinations — broader coverage of the search space than the original
n_iter=20 (250 fits instead of 360), reducing the chance of missing better-regularized trees.

In [ ]:
# Step 1: define the hyperparameter distributions for RandomizedSearchCV
# Step 2: compute and print the number of combinations and total CV fits
param_distributions = {
    'n_estimators'    : [100, 200, 500],
    'max_depth'       : [None, 10, 20, 30],
    'max_features'    : ['sqrt', 'log2'],
    'min_samples_leaf': [1, 5, 10],
}

total_combinations = (
    len(param_distributions['n_estimators']) *
    len(param_distributions['max_depth'])    *
    len(param_distributions['max_features']) *
    len(param_distributions['min_samples_leaf'])
)

print(f'Full grid size     : {total_combinations} combinations')
print(f'RandomizedSearch   : 50 sampled  (n_iter=50)')
print(f'Total model fits   : {50 * 5}  (50 combinations × 5 CV folds)')

## 5. RandomizedSearchCV — Hyperparameter Tuning

We use `RandomizedSearchCV` with `n_iter=50`, 5-fold cross-validation, and `scoring='f1'`
(per docs.md §5.4 and PHASES.md §4.1). F1 is chosen over accuracy because the dataset has
class imbalance: accuracy can be misleading when one class dominates.

The base estimator is fixed as:
- `random_state=42` — reproducibility
- `n_jobs=-1` — parallelises both tree construction and CV across all available CPU cores

In [ ]:
# Step 1: instantiate the base RandomForestClassifier with fixed random state and parallelism
# Step 2: wrap it in RandomizedSearchCV with n_iter=50, cv=5, scoring='f1'
# Step 3: fit the search on the full training set — best estimator is refit automatically
base_rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

random_search = RandomizedSearchCV(
    estimator          = base_rf,
    param_distributions= param_distributions,
    n_iter             = 50,
    cv                 = 5,
    scoring            = 'f1',
    random_state       = 42,
    n_jobs             = -1,
    verbose            = 1,
    refit              = True   # refit best model on full X_train after CV
)

print('Fitting RandomizedSearchCV (n_iter=50, cv=5) ...')
random_search.fit(X_train, y_train)
print('Done.')

## 6. Best Parameters

The randomized search selects the hyperparameter combination that maximises mean 5-fold F1
on the training set. The best estimator has been refit on the full `X_train`.

In [ ]:
# Step 1: extract the best hyperparameters, best CV score, and best fitted estimator
# Step 2: print each hyperparameter for inspection
best_params    = random_search.best_params_
best_cv_score  = random_search.best_score_
best_estimator = random_search.best_estimator_

print('Best hyperparameters:')
for k, v in best_params.items():
    print(f'  {k:20s} = {v}')
print(f'\nBest CV F1 (mean over 5 folds) : {best_cv_score:.4f}')

**Interpretation:** `max_depth` controls the complexity(h) term in Lecture 4's cost(h) =
loss(h) + λ·complexity(h). If the best `max_depth` is shallow (e.g., 10 or 20), the search
found that limiting tree depth reduces overfitting more than it increases bias — a concrete
demonstration of the regularization trade-off. `min_samples_leaf > 1` further smooths decision
boundaries by preventing leaves that represent only a handful of training points.

## 7. Predictions & Test-Set Metrics

The best estimator (refit on full `X_train`) is applied once to the held-out `X_test`. This
test set was never seen during training or CV — it is a genuine measure of generalisation.

In [ ]:
# Step 1: get predicted probabilities on the held-out test set
# Step 2: find the decision threshold that maximises F1 using the precision-recall curve
# Step 3: apply the optimal threshold to produce final predictions
# Step 4: compute all five evaluation metrics required by PHASES.md §2.3
# Step 5: compute train F1 on X_train for the Phase 6 overfitting chart
# Step 6: print a full classification report for both classes
y_proba = best_estimator.predict_proba(X_test)[:, 1]

# Find the threshold on the precision-recall curve that maximises F1
precisions_curve, recalls_curve, thresholds_curve = precision_recall_curve(y_test, y_proba)
f1s_curve = (2 * precisions_curve[:-1] * recalls_curve[:-1]
             / (precisions_curve[:-1] + recalls_curve[:-1] + 1e-9))
best_threshold = thresholds_curve[np.argmax(f1s_curve)]
print(f'Optimal threshold : {best_threshold:.4f}  (default was 0.5)')

# Apply optimal threshold instead of the default 0.5
y_pred = (y_proba >= best_threshold).astype(int)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall    = recall_score(y_test, y_pred, zero_division=0)
f1        = f1_score(y_test, y_pred, zero_division=0)
roc_auc   = roc_auc_score(y_test, y_proba)

# Train F1 for overfitting chart in Phase 6
y_train_pred = best_estimator.predict(X_train)
train_f1     = f1_score(y_train, y_train_pred, zero_division=0)
test_f1      = f1

print('=== Test-Set Metrics ===')
print(f'  Accuracy  : {accuracy:.4f}')
print(f'  Precision : {precision:.4f}')
print(f'  Recall    : {recall:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print(f'  ROC-AUC   : {roc_auc:.4f}')
print()
print(f'  Train F1  : {train_f1:.4f}  (for overfitting chart)')
print(f'  Test  F1  : {test_f1:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['On-Time (0)', 'Late (1)']))

**Interpretation:** Random Forest is a non-linear ensemble, so it should materially outperform
the linear SGDClassifier and LinearSVC models in Recall for the Late (1) class. Recall on the
Late class is the most business-critical metric: a False Negative (missed late delivery) means
logistics teams receive no alert and cannot intervene. If Train F1 >> Test F1, the best
`max_depth` is still too large — try shallower trees in a follow-up run.

## 8. Confusion Matrix

The confusion matrix breaks down predictions into True Positives (correctly flagged late),
True Negatives (correctly cleared on-time), False Positives (false alarms), and False Negatives
(missed late deliveries — the most costly error for this use case).

In [ ]:
# Step 1: compute the 2×2 confusion matrix from test predictions
# Step 2: plot using ConfusionMatrixDisplay with class labels
# Step 3: save the figure to results/ and display inline
# Step 4: print raw counts for TN, FP, FN, TP
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['On-Time (0)', 'Late (1)']
)
disp.plot(ax=ax, colorbar=False, cmap='Greens')
ax.set_title('Confusion Matrix — Random Forest', fontsize=12, pad=12)
plt.tight_layout()
plt.savefig('./results/rf_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nRaw confusion matrix:\n{cm}')
print(f'TN={cm[0,0]}  FP={cm[0,1]}  FN={cm[1,0]}  TP={cm[1,1]}')

**Interpretation:** The bottom-left cell (False Negatives) represents late orders the model
failed to flag — the most costly operational error. The top-right cell (False Positives)
represents false alarms, which cause unnecessary interventions but no delivery failures.
The business cost asymmetry (FN > FP in impact) motivates using F1 and Recall as primary
selection criteria rather than accuracy.

## 9. 5-Fold Stratified Cross-Validation

We run 5-fold stratified CV on the **training set** with the best estimator to assess score
stability. This CV is computed after the randomized search (not part of it) to provide an
independent stability estimate. High variance across folds would indicate sensitivity to the
specific data split.

In [ ]:
# Step 1: define a StratifiedKFold splitter with 5 folds, shuffled for robustness
# Step 2: run cross_val_score on the best estimator using F1 scoring
# Step 3: print fold-by-fold scores plus mean and standard deviation
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_f1_scores = cross_val_score(
    best_estimator, X_train, y_train,
    cv=cv_strategy, scoring='f1', n_jobs=-1
)

print('5-Fold Stratified CV F1 scores (training set):')
for i, s in enumerate(cv_f1_scores, 1):
    print(f'  Fold {i}: {s:.4f}')
print(f'\n  Mean  : {cv_f1_scores.mean():.4f}')
print(f'  Std   : {cv_f1_scores.std():.4f}')

**Interpretation:** Low standard deviation across folds confirms that the Random Forest's
performance is stable and does not depend strongly on which 20% of training data was withheld
for validation. This stability is a direct benefit of bagging: averaging across many trees
reduces the sensitivity to any particular data partition — exactly the variance reduction
discussed in the Lecture 4 overfitting section.

## 10. CV Score Distribution Plot

Visualising the fold-by-fold scores gives an immediate sense of stability. Wide spread indicates
high variance (sensitivity to data partition); tight spread indicates a stable estimator.

In [ ]:
# Step 1: create a bar chart of fold F1 scores
# Step 2: overlay a horizontal dashed line at the mean F1
# Step 3: save and display the figure
fig, ax = plt.subplots(figsize=(6, 3))
fold_labels = [f'Fold {i}' for i in range(1, 6)]
ax.bar(fold_labels, cv_f1_scores, color='forestgreen', alpha=0.8, edgecolor='white')
ax.axhline(
    cv_f1_scores.mean(), color='crimson', linestyle='--', linewidth=1.5,
    label=f'Mean F1 = {cv_f1_scores.mean():.4f}'
)
ax.set_ylim(0, 1)
ax.set_ylabel('F1-Score')
ax.set_title('5-Fold CV F1 Scores — Random Forest', fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig('./results/rf_cv_scores.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretation:** Each bar is one fold's F1 score on its held-out validation slice. The red
dashed line shows the mean. Consistent bar heights confirm stable performance regardless of
which training partition was used — a hallmark of ensemble models that reduce variance through
bagging.

## 11. n_estimators vs. F1 Curve

This plot shows how test-set F1 changes as the number of trees grows. Typically F1 rises
steeply for small n_estimators then plateaus — showing the point of diminishing returns
where adding more trees no longer reduces variance meaningfully. This is a concrete
illustration of Lecture 4's variance-reduction principle: more trees → lower variance → better
generalisation, up to a limit.

In [ ]:
# Step 1: define the n_estimators values to evaluate
# Step 2: for each value, train a Random Forest with the best hyperparameters
#         (except n_estimators) and compute test F1
# Step 3: plot F1 vs n_estimators and mark the best value found by RandomizedSearchCV
# Step 4: save and display the figure
n_estimator_values = [10, 25, 50, 75, 100, 150, 200, 300, 500]
f1_scores_curve = []

# Extract best params except n_estimators to hold other hyperparameters fixed
fixed_params = {k: v for k, v in best_params.items() if k != 'n_estimators'}

print('Training RF at each n_estimators value ...')
for n in n_estimator_values:
    rf_tmp = RandomForestClassifier(
        n_estimators=n,
        random_state=42,
        n_jobs=-1,
        **fixed_params
    )
    rf_tmp.fit(X_train, y_train)
    y_tmp_pred = rf_tmp.predict(X_test)
    score = f1_score(y_test, y_tmp_pred, zero_division=0)
    f1_scores_curve.append(score)
    print(f'  n_estimators={n:4d}  →  Test F1 = {score:.4f}')

# Plot the curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(n_estimator_values, f1_scores_curve, marker='o', color='forestgreen',
        linewidth=2, markersize=6, label='Test F1')
ax.axvline(
    best_params['n_estimators'], color='crimson', linestyle='--', linewidth=1.5,
    label=f'Best n_estimators = {best_params["n_estimators"]}'
)
ax.set_xlabel('n_estimators (number of trees)')
ax.set_ylabel('F1-Score (test set)')
ax.set_title('n_estimators vs. F1 — Random Forest', fontsize=12)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('./results/rf_nestimators_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nn_estimators vs. F1 curve saved.')

**Interpretation:** The curve typically shows a steep initial gain as the first dozen trees
rapidly reduce variance, followed by a plateau where additional trees produce negligible F1
improvement. This plateau effect confirms that beyond a certain B, bagging has harvested most
of the variance reduction it can provide — consistent with the theory. The red dashed line
marks the value selected by RandomizedSearchCV, which should be near or on the plateau.

## 12. Feature Importance Bar Chart (Top 15)

Random Forest provides `feature_importances_` — the mean decrease in impurity (Gini) across
all trees for each feature. This directly answers the business question: *which order
attributes drive late delivery risk most?* The top 15 features are plotted and interpreted
in the context of the supply chain domain.

In [ ]:
# Step 1: extract feature_importances_ from the best estimator
# Step 2: map importance scores to feature names
# Step 3: sort by importance descending and keep the top 15
# Step 4: build the feature_importance dict for the results pkl (all features, not just top 15)
# Step 5: plot a horizontal bar chart of the top 15 features
importances = best_estimator.feature_importances_

# Map feature names to importances
importance_series = pd.Series(importances, index=feature_names).sort_values(ascending=False)

# Full dict for pickle (all features)
feature_importance_dict = importance_series.to_dict()

# Top 15 for the chart
top15 = importance_series.head(15)

print('Top 15 feature importances:')
for feat, score in top15.items():
    print(f'  {feat:45s}  {score:.4f}')

fig, ax = plt.subplots(figsize=(9, 6))
colors = plt.cm.Greens_r(np.linspace(0.3, 0.9, len(top15)))
ax.barh(top15.index[::-1], top15.values[::-1], color=colors[::-1], edgecolor='white')
ax.set_xlabel('Mean Decrease in Impurity (Feature Importance)')
ax.set_title('Top 15 Feature Importances — Random Forest', fontsize=12)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('./results/rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretation:** Features with high importance are those that, on average, create the
largest reduction in Gini impurity at the splits where they appear across all trees. In the
supply chain context, logistics features (scheduled shipping days, shipping mode) and financial
features (discount rate, profit ratio) tend to dominate — consistent with the business
intuition that orders with tight shipping windows and heavily discounted items are more likely
to be late. One-hot encoded features (e.g., specific shipping mode categories) may appear as
several separate bars: summing their importances gives the total contribution of that original
categorical variable. This interpretability is one of the key advantages of Random Forest over
black-box models.

## 13. Summary Table

Condensed view of all key metrics for easy comparison in Phase 6.

In [ ]:
# Step 1: build a DataFrame of all key metrics and CV statistics
# Step 2: print the table in a human-readable format
summary = pd.DataFrame({
    'Metric': [
        'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC',
        'Train F1', 'Test F1', 'CV Mean F1', 'CV Std F1'
    ],
    'Value': [
        round(accuracy,            4),
        round(precision,           4),
        round(recall,              4),
        round(f1,                  4),
        round(roc_auc,             4),
        round(train_f1,            4),
        round(test_f1,             4),
        round(cv_f1_scores.mean(), 4),
        round(cv_f1_scores.std(),  4),
    ]
})
print(summary.to_string(index=False))

**Interpretation:** This table condenses all evaluation dimensions into one view. The Train/Test
F1 pair reveals the bias-variance balance: a small gap indicates well-calibrated regularization
(via `max_depth` and `min_samples_leaf`); a large gap indicates the trees are still overfitting
their bootstrap samples. ROC-AUC is threshold-independent and complements F1 (which depends on
the 0.5 decision threshold). Both metrics appear in the Phase 6 five-model comparison table.

## 14. Save Results

All outputs are saved to `results/rf_results.pkl` using the exact schema defined in PHASES.md
§2.3. Phase 6 loads this file directly — any key mismatch will break the merge notebook.
The schema is verified with assertions before pickling.

In [ ]:
# Step 1: assemble the results dict with all keys required by PHASES.md §2.3
# Step 2: assert that the key set matches the schema exactly
# Step 3: assert that the nested metrics dict has the correct keys
# Step 4: pickle and save to results/rf_results.pkl
results = {
    'model_name'        : 'RandomForest',
    'model'             : best_estimator,
    'best_params'       : best_params,
    'y_pred'            : y_pred,
    'y_proba'           : y_proba,
    'metrics'           : {
        'accuracy'  : accuracy,
        'precision' : precision,
        'recall'    : recall,
        'f1'        : f1,
        'roc_auc'   : roc_auc,
    },
    'cv_f1_scores'      : cv_f1_scores,
    'train_f1'          : train_f1,
    'test_f1'           : test_f1,
    'feature_importance': feature_importance_dict,  # dict {feature_name: importance_score}
}

# Verify schema before pickling
required_keys = {
    'model_name', 'model', 'best_params', 'y_pred', 'y_proba',
    'metrics', 'cv_f1_scores', 'train_f1', 'test_f1', 'feature_importance'
}
assert set(results.keys()) == required_keys, (
    f'Schema mismatch: {set(results.keys()) ^ required_keys}'
)

required_metric_keys = {'accuracy', 'precision', 'recall', 'f1', 'roc_auc'}
assert set(results['metrics'].keys()) == required_metric_keys, 'Metrics schema mismatch'

with open(RESULTS_PATH, 'wb') as f:
    pickle.dump(results, f)

print(f'Results saved to: {RESULTS_PATH}')
print(f'Keys: {list(results.keys())}')
print(f'Metrics: {results["metrics"]}')
print(f'feature_importance entries: {len(feature_importance_dict)}')

**Verification:** The assertions confirm that the saved dict has exactly the keys required by
PHASES.md §2.3. Unlike the linear models, `feature_importance` is **not** None here — it
is a dict mapping every feature name to its mean decrease in impurity score. Phase 6 will
use this to reproduce the feature importance chart in the merged notebook.

## 15. Conclusion

This notebook implemented Section 5.4 of `docs.md`:

- **Model:** `RandomForestClassifier(random_state=42, n_jobs=-1)` — a bagging ensemble of
  decision trees, the first of the two allowed out-of-class extensions
- **Tuning:** `RandomizedSearchCV(n_iter=20, cv=5, scoring='f1', random_state=42, n_jobs=-1)`
  over the four-hyperparameter grid from docs.md §5.4
- **Evaluation:** confusion matrix, accuracy / precision / recall / F1 / ROC-AUC on the
  held-out test set, train F1 for the Phase 6 overfitting chart, and 5-fold stratified CV F1
- **Extra plots:** n_estimators vs. F1 curve (Fig 9 of docs.md §7.3.B), and top-15 feature
  importance bar chart (Fig 10) with business-domain interpretation
- **Lecture 4 connection:** `max_depth` and `min_samples_leaf` tied to complexity(h) in
  cost(h) = loss(h) + λ·complexity(h); bagging tied to Lecture 4's overfitting discussion
- **Output:** `results/rf_results.pkl` with the exact schema required by PHASES.md §2.3,
  including a populated `feature_importance` dict for use in Phase 6

**Expected scientific role:** As a bagging ensemble with non-linear boundaries, Random Forest
should outperform the three linear models (SGDClassifier, LinearSVC) and provide a strong
baseline for comparison against XGBoost (boosting). Its feature importances also provide the
most direct business insight into *why* orders are predicted late.